In [1]:
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline
from FairGen.src.models.fairgen_adapter import FairGenLayer, FairGenNetwork
from FairGen.src.models.io import load_state_dict
from FairGen.src.configs.config import parse_precision

# ================= CONFIGURATION =================
BASE_MODEL = "CompVis/stable-diffusion-v1-4"
DEVICE = "cuda"
PRECISION = "fp32" # Use fp16 if your GPU is limited

# PARAMETERS TO TEST
PROMPT = "A CEO" #"a photo of the face of a CEO, a person" # Paper recommendation [cite: 241]
NEGATIVE_PROMPT = ""
GUIDANCE_SCALE = 7.5
# =================================================

def generate_test_image(seed=42):
    weight_dtype = parse_precision(PRECISION)
    
    # 1. Load Base Model
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL, torch_dtype=weight_dtype
    ).to(DEVICE)
    pipe.safety_checker = None

    for seed in seeds:
        # 4. Generate
        # generator = torch.Generator(device=DEVICE).manual_seed(SEED)
        generator = torch.Generator(device=DEVICE).manual_seed(seed)

        
        # print(f"Generating image with Scale: {ADAPTER_SCALE}, Seed: {SEED}...")
        print(f"Generating base images, Seed: {seed}...")
        image = pipe(
            prompt=PROMPT,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=50,
            guidance_scale=GUIDANCE_SCALE,
            generator=generator
        ).images[0]

        # 5. Save
        # save_name = f"test_scale_{ADAPTER_SCALE}_seed_{SEED}.png"
        save_name = f"ceo_output_baseline/basic_prompt_test_seed_{seed}.png"
        image.save(save_name)
        print(f"Saved to {save_name}")

if __name__ == "__main__":
    seeds = [1,2,3,4,5,6,7,8,9,10]
    generate_test_image(seeds)

c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu118 with CUDA 1108 (you have 2.7.1+cu118)
    Python  3.10.11 (you have 3.10.19)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=

Generating base images, Seed: 5...


100%|██████████| 50/50 [00:13<00:00,  3.74it/s]


Saved to ceo_output_baseline/basic_prompt_test_seed_5.png
Generating base images, Seed: 6...


100%|██████████| 50/50 [01:22<00:00,  1.66s/it]


...

Saved to ceo_output_baseline/basic_prompt_test_seed_8.png
Generating base images, Seed: 9...


100%|██████████| 50/50 [01:23<00:00,  1.66s/it]


Saved to ceo_output_baseline/basic_prompt_test_seed_9.png
Generating base images, Seed: 10...


100%|██████████| 50/50 [01:22<00:00,  1.66s/it]


Saved to ceo_output_baseline/basic_prompt_test_seed_10.png


In [ ]:
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline
from FairGen.src.models.fairgen_adapter import FairGenLayer, FairGenNetwork
from FairGen.src.models.io import load_state_dict
from FairGen.src.configs.config import parse_precision

# ================= CONFIGURATION =================
BASE_MODEL = "CompVis/stable-diffusion-v1-4"
ADAPTER_PATH = "FairGen/output/gender/female/gender_female_1000steps.safetensors"#"FairGen/output/gender/female/rank_8_gender_female_500steps.safetensors"
DEVICE = "cuda"
PRECISION = "fp32" # Use fp16 if your GPU is limited

# PARAMETERS TO TEST
PROMPT = "A CEO" #"a photo of the face of a CEO, a person" # Paper recommendation [cite: 241]
NEGATIVE_PROMPT = ""
# SEED = 2
ADAPTER_SCALE = 7 #0.3 #0.3  # Try 0.3 (Paper default) vs 1.0 (Training default) 
GUIDANCE_SCALE = 7.5
# =================================================

def generate_test_image(seeds=[42]):
    weight_dtype = parse_precision(PRECISION)
    
    # 1. Load Base Model
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL, torch_dtype=weight_dtype
    ).to(DEVICE)
    pipe.safety_checker = None
    
    # 2. Load and Scale Adapter
    # Load the raw weights from the safetensors file
    adapter_state, metadata = load_state_dict(ADAPTER_PATH, weight_dtype)
    rank = int(float(metadata.get("rank", 1)))
    alpha = float(metadata.get("alpha", 1))
    
    print(f"Loaded adapter: {ADAPTER_PATH} (Rank: {rank}, Alpha: {alpha})")

    # Manually apply scaling to the weights before loading into the network
    # This ensures we know EXACTLY what multiplier is being applied.
    scaled_state = {}
    for key, tensor in adapter_state.items():
        if ".lora_up." in key or ".lora_down." in key:
            # We apply ADAPTER_SCALE here. 
            # Note: If FairGenLayer handles alpha/rank internally, 
            # scaling here effectively acts as the 'multiplier'.
            scaled_state[key] = tensor * ADAPTER_SCALE
        else:
            scaled_state[key] = tensor

    # 3. Setup FairGen Network
    network = FairGenNetwork(
        pipe.unet,
        rank=rank,
        alpha=alpha,
        module=FairGenLayer,
        cross_attention_only=True,
    ).to(DEVICE, dtype=weight_dtype)
    
    network.load_state_dict(scaled_state)

    # 4. Generate
    # generator = torch.Generator(device=DEVICE).manual_seed(SEED)
    for seed in seeds:
        generator = torch.Generator(device=DEVICE).manual_seed(seed)

        
        # print(f"Generating image with Scale: {ADAPTER_SCALE}, Seed: {SEED}...")
        print(f"Generating image with Scale: {ADAPTER_SCALE}, Seed: {seed}...")
        with network:
            image = pipe(
                prompt=PROMPT,
                negative_prompt=NEGATIVE_PROMPT,
                num_inference_steps=50,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator
            ).images[0]

        # 5. Save
        # save_name = f"test_scale_{ADAPTER_SCALE}_seed_{SEED}.png"
        save_name = f"ceo_output_testing/1k_adapter_basic_prompt_test_scale_{ADAPTER_SCALE}_seed_{seed}.png"
        image.save(save_name)
        print(f"Saved to {save_name}")

if __name__ == "__main__":
    seeds =[1,2,3,4,5,6,7,8,9,10]
    generate_test_image(seeds)

c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu118 with CUDA 1108 (you have 2.7.1+cu118)
    Python  3.10.11 (you have 3.10.19)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=

FairGen/output/gender/female/gender_female_1000steps.safetensors
Loaded adapter: FairGen/output/gender/female/gender_female_1000steps.safetensors (Rank: 1, Alpha: 1.0)
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_t

100%|██████████| 50/50 [00:13<00:00,  3.67it/s]


Saved to ceo_output_testing/1k_adapter_basic_prompt_test_scale_7_seed_1.png
Generating image with Scale: 7, Seed: 2...


100%|██████████| 50/50 [00:13<00:00,  3.68it/s]


Saved to ceo_output_testing/1k_adapter_basic_prompt_test_scale_7_seed_2.png
Generating image with Scale: 7, Seed: 3...


100%|██████████| 50/50 [00:13<00:00,  3.73it/s]


Saved to ceo_output_testing/1k_adapter_basic_prompt_test_scale_7_seed_3.png
Generating image with Scale: 7, Seed: 4...


 20%|██        | 10/50 [00:02<00:10,  3.71it/s]

In [ ]:
import torch
from pathlib import Path
from diffusers import StableDiffusionPipeline
from FairGen.src.models.fairgen_adapter import FairGenLayer, FairGenNetwork
from FairGen.src.models.io import load_state_dict
from FairGen.src.configs.config import parse_precision

# ================= CONFIGURATION =================
BASE_MODEL = "CompVis/stable-diffusion-v1-4"
ADAPTER_PATH = "FairGen/output/gender/female/rank_8_gender_female_500steps.safetensors" #gender_female_1000steps.safetensors"
DEVICE = "cuda"
PRECISION = "fp32" 

# PARAMETERS
# Use the full anchor template from the paper [cite: 219, 406]
PROMPT = "a photo of the face of a CEO, a person" 
NEGATIVE_PROMPT = ""
# Recommended alpha from paper is 0.3 
ADAPTER_SCALE = 0.3 
GUIDANCE_SCALE = 7.5
# =================================================

def generate_test_image(seeds=[1,2,3,4,5,6,7,8,9,10]):
    weight_dtype = parse_precision(PRECISION)
    
    # 1. Load Base Model
    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL, torch_dtype=weight_dtype
    ).to(DEVICE)
    pipe.safety_checker = None
    
    # 2. Load Adapter State (No manual scaling here) 
    adapter_state, metadata = load_state_dict(ADAPTER_PATH, weight_dtype)
    rank = int(float(metadata.get("rank", 1)))
    
    # The paper uses alpha=0.3 at inference 
    # We pass this to the network instead of scaling the state dict
    alpha = ADAPTER_SCALE 
    
    print(f"Loaded adapter: {ADAPTER_PATH} (Rank: {rank}, Alpha/Scale: {alpha})")

    # 3. Setup FairGen Network
    # Ensure it only targets cross-attention layers 
    network = FairGenNetwork(
        pipe.unet,
        rank=rank,
        alpha=alpha,
        module=FairGenLayer,
        cross_attention_only=True,
    ).to(DEVICE, dtype=weight_dtype)
    
    # Load the original unscaled weights [cite: 188, 189]
    network.load_state_dict(adapter_state)

    # 4. Generate
    Path("ceo_output_testing").mkdir(parents=True, exist_ok=True)
    
    for seed in seeds:
        generator = torch.Generator(device=DEVICE).manual_seed(seed)
        print(f"Generating image | Scale: {alpha} | Seed: {seed}...")
        
        # The 'with network:' context should apply the h*M * alpha update [cite: 164, 189]
        with network:
            image = pipe(
                prompt=PROMPT,
                negative_prompt=NEGATIVE_PROMPT,
                num_inference_steps=50,
                guidance_scale=GUIDANCE_SCALE,
                generator=generator
            ).images[0]

        # 5. Save
        save_name = f"ceo_output_testing/1k_adapter_test_scale_{alpha}_seed_{seed}.png"
        image.save(save_name)
        print(f"Saved to {save_name}")

if __name__ == "__main__":
    generate_test_image()

c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu118 with CUDA 1108 (you have 2.7.1+cu118)
    Python  3.10.11 (you have 3.10.19)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=

FairGen/output/gender/female/rank_8_gender_female_500steps.safetensors
Loaded adapter: FairGen/output/gender/female/rank_8_gender_female_500steps.safetensors (Rank: 8, Alpha/Scale: 0.3)
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_1_attentions_0_transforme

100%|██████████| 50/50 [00:13<00:00,  3.68it/s]


Saved to ceo_output_testing/1k_adapter_test_scale_0.3_seed_1.png
Generating image | Scale: 0.3 | Seed: 2...


100%|██████████| 50/50 [00:13<00:00,  3.67it/s]


...

100%|██████████| 50/50 [00:18<00:00,  2.74it/s]


Saved to ceo_output_testing/1k_adapter_test_scale_0.3_seed_6.png
Generating image | Scale: 0.3 | Seed: 7...


100%|██████████| 50/50 [00:20<00:00,  2.42it/s]


Saved to ceo_output_testing/1k_adapter_test_scale_0.3_seed_7.png
Generating image | Scale: 0.3 | Seed: 8...


  8%|▊         | 4/50 [00:02<00:28,  1.63it/s]

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from FairGen.src.models.fairgen_adapter import FairGenLayer, FairGenNetwork
from FairGen.src.models.io import load_state_dict

# ================= CONFIGURATION =================
ADAPTER_PATH = "FairGen/output/gender/female/gender_female_1000steps.safetensors"
ADAPTER_SCALE = 30 # Paper standard for Rank 1 [cite: 218]
PROMPT = "a photo of the face of a CEO, a person" # Crucial anchor 
# =================================================

def generate_fairgen_image(seeds=[1, 2, 3]):
    # 1. Load Pipe
    pipe = StableDiffusionPipeline.from_pretrained("CompVis/stable-diffusion-v1-4").to("cuda")
    
    # 2. Load specialized Rank 1 Weights
    # Do NOT manually scale the state_dict tensors. 
    state_dict, metadata = load_state_dict(ADAPTER_PATH, torch.float32)
    rank = 1 # Force Rank 1 logic [cite: 142]
    
    # 3. Initialize the Specialized FairGen Network
    # This replaces standard Linear layers with the p/q vector logic [cite: 143, 145]
    network = FairGenNetwork(
        pipe.unet,
        rank=rank,
        alpha=ADAPTER_SCALE, # Alpha is applied here: W + alpha * (qT*x)*p 
        module=FairGenLayer,
        cross_attention_only=True
    ).to("cuda")
    
    network.load_state_dict(state_dict)

    # 4. Generate with the 'with' context to active the p/q math [cite: 161]
    for seed in seeds:
        generator = torch.Generator("cuda").manual_seed(seed)
        with network:
            image = pipe(
                prompt=PROMPT,
                num_inference_steps=50,
                guidance_scale=7.5,
                generator=generator
            ).images[0]
        
        image.save(f"rank1_test_seed_{seed}.png")
        print(f"Generated seed {seed} with Rank 1 adapter.")

if __name__ == "__main__":
    generate_fairgen_image()

c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\User\anaconda3\envs\fairgen\lib\site-packages\accelerate\utils\torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu118 with CUDA 1108 (you have 2.7.1+cu118)
    Python  3.10.11 (you have 3.10.19)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=

FairGen/output/gender/female/gender_female_1000steps.safetensors
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_1_attentions_0_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_1_attentions_1_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_1_at

100%|██████████| 50/50 [00:13<00:00,  3.58it/s]


Generated seed 1 with Rank 1 adapter.


  2%|▏         | 1/50 [00:03<02:41,  3.30s/it]

In [3]:
import torch
from diffusers import DiffusionPipeline
from safetensors.torch import load_file
import os

# 1. Configuration
base_model_id = "runwayml/stable-diffusion-v1-5" #"CompVis/stable-diffusion-v1-4"
# This matches the --exp_name you used in training
# weights_path = "unified-concept-editing/uce_models/debias_sd14.safetensors"
# weights_path = "unified-concept-editing/uce_models/debias_sd15_balanced_race_doctor.safetensors" #"unified-concept-editing/uce_models/debias_sd15_mainly_female_doctor.safetensors" #"unified-concept-editing/uce_models/debias_sd14_fixed.safetensors" 
# weights_path = "unified-concept-editing/uce_models/debias_sd15_balanced_white_black_race_doctor.safetensors"
weights_path = "unified-concept-editing/uce_models/debias_sd15_balanced_white_black_male_female_doctor.safetensors"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model: {base_model_id}...")
# Load the standard Stable Diffusion model
pipe = DiffusionPipeline.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
).to(device)

# 2. Inject your Debiased Weights
if os.path.exists(weights_path):
    print(f"Injecting debiased weights from: {weights_path}")
    # Load the specific weights you trained
    uce_weights = load_file(weights_path)
    
    # Apply them to the model (strict=False is crucial because we are only updating specific layers)
    pipe.unet.load_state_dict(uce_weights, strict=False)
    print("Weights injected successfully!")
else:
    print(f"ERROR: Could not find {weights_path}. Did you name your experiment 'debias_sd14'?")
    exit()

# 3. Test Prompts
prompts = [
    "A photo of a male Doctor, person",
    # "A photo of a Nurse, person",
    # "A portrait of a Carpenter, person",
    # "A photo of a Construction Worker, person",
    # "A portrait of a Firefighter, person" # Control test: You didn't debias this one!
]

# 4. Generate Images
print("\nGenerating images...")
os.makedirs("test_outputs", exist_ok=True)

for prompt in prompts:
    print(f"Generating: {prompt}")
    # Generate 4 images per prompt to check consistency
    images = pipe(prompt, num_images_per_prompt=20, num_inference_steps=30).images
    
    # Save them
    clean_name = prompt.replace(" ", "_").replace("A_portrait_of_a_", "").replace("A_photo_of_a_", "wb_")
    for i, img in enumerate(images):
        img.save(f"test_outputs/{clean_name}_{i}.png")

print("\nDone! Check the 'test_outputs' folder.")

Loading base model: runwayml/stable-diffusion-v1-5...


text_encoder\model.safetensors not found
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


Injecting debiased weights from: unified-concept-editing/uce_models/debias_sd15_balanced_white_black_male_female_doctor.safetensors
Weights injected successfully!

Generating images...
Generating: A photo of a male Doctor, person


100%|██████████| 30/30 [00:44<00:00,  1.48s/it]



Done! Check the 'test_outputs' folder.


In [ ]:
# python trainscripts/uce_sd_debias.py --edit_concepts "Doctor" --debias_concepts "male; female" --desired_ratios 0.1 0.9 --preserve_concepts "person; human; portrait" --exp_name "debias_sd15_mainly_female_doctor" --model_id "runwayml/stable-diffusion-v1-5"

# python trainscripts/uce_sd_debias.py --edit_concepts "Doctor" --debias_concepts "white; black; american indian; native american; asian" --desired_ratios 0.2 0.2 0.2 0.2 0.2 --preserve_concepts "person; human; portrait" --exp_name "debias_sd15_balanced_race_doctor" --model_id "runwayml/stable-diffusion-v1-5"

# python trainscripts/uce_sd_debias.py --edit_concepts "Doctor" --debias_concepts "white; black; asian" --desired_ratios 0.33 0.33 0.33 --preserve_concepts "person; human; portrait" --exp_name "debias_sd15_balanced_race_doctor" --model_id "runwayml/stable-diffusion-v1-5"

python trainscripts/uce_sd_debias.py --edit_concepts "Doctor" --debias_concepts "male; female; white; black; asian" --desired_ratios 0.5 0.5 0.33 0.33 0.33 --preserve_concepts "person; human; portrait" --exp_name "debias_sd15_balanced_race_doctor" --model_id "runwayml/stable-diffusion-v1-5"

In [2]:
import torch
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm
from PIL import Image

# Diffusers & Transformers
from diffusers import StableDiffusionPipeline, DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from transformers import CLIPTextModel, CLIPTokenizer
from torch.utils.data import Dataset, DataLoader
from peft import LoraConfig, get_peft_model

lora_path = r"C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\SDModelDebiasing\outputs\lora_debiased\checkpoint-epoch-5"
prompt = r"a photo of a doctor"
print(f"Loading LoRA from {lora_path} for testing...")

# Load Base
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

# Load Adapter
pipe.load_lora_weights(lora_path)

print("Generating images...")
images = []
num_images = 20
for i in range(num_images):
    img = pipe(prompt, num_inference_steps=30).images[0]
    img.save(f"outputs/test_result_{i}.png")
    images.append(img)
print("Test images saved.")

Loading LoRA from C:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\SDModelDebiasing\outputs\lora_debiased\checkpoint-epoch-5 for testing...


Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.58it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
No LoRA keys associated to UNet2DConditionModel found with the prefix='unet'. This is safe to ignore if LoRA state dict didn't originally have any UNet2DConditionModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an i

Generating images...


100%|██████████| 30/30 [00:02<00:00, 10.05it/s]


Test images saved.


In [3]:
from diffusers import StableDiffusionPipeline
from peft import PeftModel
import torch

# Load baseline
print("Loading baseline...")
baseline = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

img_baseline = baseline(
    "a photo of a doctor",
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]
img_baseline.save("test_baseline.png")

# Load with LoRA
print("Loading with LoRA...")
lora_pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
)

lora_pipe.unet = PeftModel.from_pretrained(
    lora_pipe.unet,
    "outputs/lora_debiased/checkpoint-epoch-5"
)
lora_pipe = lora_pipe.to("cuda")

img_lora = lora_pipe(
    "a photo of a doctor",
    generator=torch.Generator("cuda").manual_seed(42)
).images[0]
img_lora.save("test_lora.png")

print("✓ Compare test_baseline.png and test_lora.png")
print("They should look DIFFERENT if LoRA is working!")

Couldn't connect to the Hub: (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /api/models/runwayml/stable-diffusion-v1-5 (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 63c7345e-57b8-43f5-bc17-7f4af3f3a8ae)').
Will try to load from local cache.


Loading baseline...


Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  4.67it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
100%|██████████| 50/50 [00:04<00:00, 10.30it/s]


Loading with LoRA...


Loading pipeline components...: 100%|██████████| 6/6 [00:01<00:00,  5.76it/s]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .
100%|██████████| 50/50 [00:05<00:00,  8.53it/s]


✓ Compare test_baseline.png and test_lora.png
They should look DIFFERENT if LoRA is working!


In [ ]:
python -m diffusers.pipelines.stable_diffusion.convert_from_ckpt --model_path runwayml/stable-diffusion-v1-5 --checkpoint_path sd-v1-5.ckpt --original_config_file models/sd/configs/stable-diffusion/v1-inference.yaml


In [ ]:
# python generation.py --config="models/sd/configs/stable-diffusion/v1-inference.yaml" --ckpt="models/sd/models/ldm/stable-diffusion-v1/model.ckpt" --plms --attr-list="Skin_tone" --outdir="./ckpts/a_headshot_of_a_person_Skin_tone/original_prompt_embedding/sample_results" --prompt-path="./ckpts/a_headshot_of_a_personSkin_tone/original_prompt_embedding/basis_final_embed_19.pt" --n_iter=5 --n_rows=5 --n_samples=1